# Reference Resolution Dev Notebook

In [11]:
import random
import torch
import io
import pyarrow as pa
import os
import copy
import pytorch_lightning as pl
from sacred import Experiment
from PIL import Image
from tqdm.auto import tqdm
import numpy as np
import skimage.io as skio
import matplotlib.pyplot as plt
from refer import REFER
import pandas as pd

from torch.optim import AdamW

from transformers import ElectraTokenizer

from refcoco_utils import get_bounded_subimage
from refcoco_utils import _config
from refcoco_utils import _loss_names

from meter.transforms import keys_to_transforms
from meter.config import ex
from meter.modules import METERTransformerSS
from meter.datamodules.multitask_datamodule import MTDataModule
from meter.datasets.base_dataset import BaseDataset

## RefCOCO Data

### Utility Functions

### Import  Data

In [12]:
data_root = '/home/claytonfields/nlp/code/data/coco'  # contains refclef, refcoco, refcoco+, refcocog and images
dataset = 'refcoco' 
splitBy = 'unc'
refer = REFER(data_root, dataset, splitBy)

loading dataset refcoco into memory...
testing
creating index...
index created.
DONE (t=10.14s)


In [13]:
refer.IMAGE_DIR = '/home/claytonfields/nlp/code/data/coco/images/mscoco/train2014'

## METER Model

In [14]:
_config = copy.deepcopy(_config)
pl.seed_everything(_config["seed"])

# dm = MTDataModule(_config, dist=False)
model = METERTransformerSS(_config)
# exp_name = f'{_config["exp_name"]}'
# os.makedirs(_config["log_dir"], exist_ok=True)

Global seed set to 0
Some weights of the model checkpoint at google/electra-small-discriminator were not used when initializing ElectraModel: ['discriminator_predictions.dense_prediction.bias', 'discriminator_predictions.dense_prediction.weight', 'discriminator_predictions.dense.bias', 'discriminator_predictions.dense.weight']
- This IS expected if you are initializing ElectraModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing ElectraModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


## To Do: Write New Data Class for Ref Res with multiple samples

1. Deliver single sentence with sub-images and text_labels, masks and ids so that METER can perform all subimage at once.

2.  May require padding to max_num_bb = 75

In [15]:
class RefcocoDataset(torch.utils.data.Dataset):

    def __init__(self, refer, tokenizer, split='', max_bb = 75):
        self.tokenizer = tokenizer
        self.refer = refer
        self.max_bb = max_bb
        self.split = split
        self.sent_ids = self.get_sent_ids()
        self.duds = []
        

    def __len__(self):
        return len(self.sent_ids)
    
    def get_sent_ids(self):
        sent_ids = []
        for ref_id in self.refer.getRefIds(split=self.split):
            ref = self.refer.Refs[ref_id]
            for sent_id in ref['sent_ids']:
                sent_ids.append(sent_id)
        return sent_ids

    def __getitem__(self, index):
        sent_id = self.sent_ids[index]
        ref = self.refer.sentToRef[sent_id]
        sent = self.refer.Sents[sent_id]
        
        img_id = ref['image_id']
        ann_id = ref['ann_id']
        objs = self.refer.imgToAnns[img_id]
        obj_ids = [obj['id'] for obj in objs]
        
        sub_images = []
        for obj in objs:
            try:
                x_a = get_bounded_subimage(refer, img_id, obj['id'], xs=224,ys=224, show=False)
            except ValueError:
                print(f'ValueError at setence id: {sent_id}')
                self.duds.append(sent_id)
                break
                
            if x_a is not None:
                sub_images.append(x_a)
        num_sub_images = len(sub_images)      
            
        text_ids = tokenizer.encode(
            sent['sent'],
            padding="max_length",
            truncation=True,
            max_length=40,
            return_special_tokens_mask=True,
        )
        text_masks = [1 if text_ids[i]>0 else 0 for i,_ in enumerate(text_ids)]
        text_labels = [[-100 for i in range(40)]]

        ids = [text_ids for i in range(num_sub_images)]
        masks = [text_masks for _ in range(num_sub_images)]
        labels = [text_labels for i in range(num_sub_images)]
            
        return_dict = {
            'ann_id' : ann_id,
            'image' : [torch.cat(sub_images)],
            'obj_ids' : obj_ids,
            'sent_id' : sent_id,
            'text' : sent['sent'],
            'text_ids' : torch.tensor(ids),
            'text_labels' : torch.tensor(labels),
            'text_masks' : torch.tensor(masks)
        }  

        return return_dict

## Ref Res with METER

In [16]:
optimizer = AdamW(model.parameters(), lr=1e-4)
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')

# Ref Res with METER
tokenizer = ElectraTokenizer.from_pretrained('google/electra-small-discriminator')
BATCH_SIZE = 1

epochs = 1
# loader = dm.train_dataloader()
optim = AdamW(model.parameters(), lr=1e-4)
loss_fn = torch.nn.functional.cross_entropy
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')

In [17]:
train_ds = RefcocoDataset(refer, tokenizer, split='train')
# train_ds = torch.utils.data.Subset(ds, sent_ids[729:])


In [18]:
train_params = {'batch_size': BATCH_SIZE,
                'shuffle': False,
                'num_workers': 0
                }

training_loader = torch.utils.data.DataLoader(train_ds, **train_params)

In [19]:
eval_ds = RefcocoDataset(refer, tokenizer, split='val')
eval_ds.__len__()

10834

In [20]:
eval_params = {'batch_size': BATCH_SIZE,
                'shuffle': True,
                'num_workers': 0
                }
eval_loader = torch.utils.data.DataLoader(eval_ds, **eval_params)

In [15]:
def train(model, training_ds, optimizer, loss_fn, device):
    model.to(device)
    model.train()
    losses = []
    for data in tqdm(training_ds):
        if data['sent_id'] in training_ds.duds:
            continue
        try:
            optimizer.zero_grad()
            
            sent_id = data['sent_id']
            infer_dict = model.infer(data)
            logits = model.ref_classifier(infer_dict['cls_feats'])

            obj_ids = data['obj_ids']
            ann_id = data['ann_id']

            target = torch.tensor([obj_ids.index(ann_id)])
            loss = loss_fn(logits.reshape(1,-1),target)
            losses.append(loss.item())
            loss.backward()

            optimizer.step()
        except RuntimeError:
            print(f'Runtime Error at sent_id = {sent_id}')
            training_ds.duds.append(sent_id)
    return losses, loss

def evaluate(model, eval_ds):
    gold = []
    with torch.no_grad():
        for data in tqdm(eval_ds):
            if data['sent_id'] in training_ds.duds:
                continue
            try:
                sent_id = data['sent_id']
                infer_dict = model.infer(data)
                logits = model.ref_classifier(infer_dict['cls_feats'])

                obj_ids = data['obj_ids']
                ann_id = data['ann_id']

                pred_index = np.argmax(logits)
                pred_id = obj_ids[pred_index]
                target = torch.tensor([obj_ids.index(ann_id)])
                if pred_id == ann_id:
                    gold.append(1)
                else:
                    gold.append(0)
            except RuntimeError:
                print(f'RuntimeError at sent_id = {sent_id}')
                eval_ds.duds.append(sent_id)
    return gold

In [21]:
for epoch in range(epochs):
    losses, loss = train(model, train_ds, optimizer, loss_fn, device)
    print(f'Epoch: {epoch}, Loss:  {loss.item()}')  
    loss_frame = pd.DataFrame(losses,columns=['Loss'])
    gold = evaluate(model, eval_ds)
    acc = np.average(gold)
    print(f'acurracy on test set {acc}')

NameError: name 'train' is not defined

In [26]:
infer = model(train_ds[0])

In [28]:
infer['cls_feats'].shape

torch.Size([33, 512])

In [31]:
logits = model.ref_classifier(infer['cls_feats'])

In [35]:
obj_ids = train_ds[0]['obj_ids']
ann_id = train_ds[0]['ann_id']

target = torch.tensor([obj_ids.index(ann_id)])
loss = loss_fn(logits.reshape(1,-1),target)

In [36]:
loss

tensor(3.7166, grad_fn=<NllLossBackward>)

In [ ]:
loss_fn(logits, )

In [63]:
losses

[3.465179920196533,
 3.493135452270508,
 3.48661732673645,
 3.502887010574341,
 3.4891436100006104]

#### Full Training Loop

In [ ]:
import matplotlib.pyplot as plt
plt.plot(losses)

In [ ]:
target

In [52]:
losses = [i for i in range(30)]
csv_string = f'Epoch_{epoch}_losses.csv'
pd.DataFrame(losses,columns=['Loss'])#.to_csv(csv_string)

,Loss
0,0
1,1
2,2
3,3
4,4
5,5
6,6
7,7
8,8
9,9


## Test Cells:

In [ ]:
refer.sentToRef[sent_id]